# Ilham on a free Kaggle GPU

Runs **Ollama** (her brain) on Kaggle's free dual T4 (32 GB total) and exposes it through a free Cloudflare tunnel, so a slow PC can still use a big model.

**Before running:** in the right panel set *Accelerator* = **GPU T4 x2** and *Internet* = **On** (needs a phone-verified Kaggle account).

Run the cell, wait ~2 min, then copy the printed lines into Ilham's **⌨ Console** on your PC:
```
/set ollamaurl https://...trycloudflare.com
/set model qwen2.5:14b
/model
```
Keep the cell running while you chat. Run **only one** Ilham (desktop *or* phone) against the tunnel at a time.

| Model | VRAM | Notes |
|---|---|---|
| `qwen2.5:7b`  | ~6 GB  | fast, safe default |
| `qwen2.5:14b` | ~10 GB | **recommended** — sharp Urdu + reasoning, one T4 |
| `qwen2.5:32b` | ~20 GB | smartest; uses both T4s (spread is enabled below) |


In [ ]:
import os, re, subprocess, time

MODEL = 'qwen2.5:14b'   # recommended. 'qwen2.5:7b' = faster, 'qwen2.5:32b' = smartest (both T4s)

# Kaggle's image lacks zstd, which Ollama's installer now needs.
subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Serve on all interfaces; keep the model resident all session; allow spreading big models across both T4s.
env = dict(os.environ,
           OLLAMA_HOST='0.0.0.0:11434',
           OLLAMA_ORIGINS='*',
           OLLAMA_KEEP_ALIVE='24h',
           OLLAMA_SCHED_SPREAD='1')
subprocess.Popen(['ollama', 'serve'], env=env)
time.sleep(6)

subprocess.run(['ollama', 'pull', MODEL], check=True)
print('\nModel ready:', MODEL)

# Pre-warm: load the model into the GPUs now, so your FIRST message replies instantly (no 50s cold start).
print('Warming up the model into the GPUs...')
subprocess.run(['ollama', 'run', MODEL, 'Say ready.'], check=True)
print('Warm and loaded.\n')

# Public tunnel so your PC can reach this Ollama over the internet.
subprocess.run('curl -sL -o cloudflared '
               'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
               '&& chmod +x cloudflared', shell=True, check=True)
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tun.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n\n==============================================')
        print('  Paste into Ilham Console on your PC:')
        print('     /set ollamaurl ' + m.group(0))
        print('     /set model ' + MODEL)
        print('     /model')
        print('  Keep this cell running while you chat.')
        print('  Run only ONE Ilham against this tunnel at a time.')
        print('==============================================\n')
